<a href="https://colab.research.google.com/github/ADRAKECROWDER/AIML2003-nlp/blob/main/Week2/HowMachinesRead.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook compares how machines represent text and images as numbers. The first half uses TF-IDF to retrieve similar movie reviews, and the second half uses HOG to retrieve similar images. The goal was to see where each method worked, where each one failed, what those failures revealed and how they compared.

In [1]:
# Cell 1 Setup
!pip install -q nltk scikit-learn scikit-image tensorflow

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
from nltk.corpus import movie_reviews
import nltk
from sklearn.manifold import TSNE
import tensorflow as tf
from skimage.feature import hog
from skimage.color import rgb2gray
from skimage.transform import resize
from collections import Counter

nltk.download('movie_reviews', quiet=True)

True

Cell 2: Loads Corpus
This cell loads the movie review corpus and organizes it into matching lists: one for the review text and one for the label. The output confirms that all 2,000 reviews load correctly and shows a short excerpt so the text can be checked before building feature vectors.

In [2]:
# Cell 2 Load the corpus with random starting point
import random

# 1. Access all available file identifiers in the movie_reviews corpus
all_fileids = movie_reviews.fileids()

# 2. Determine the total count of files (should be 2000)
total_files = len(all_fileids)

# 3. Use random.randint(a, b) to select a random integer N where 0 <= N <= total_files - 1
# This serves as our dynamic entry point into the list
start_idx = random.randint(0, total_files - 1)

# 4. Use list slicing and concatenation to 'reorder' the list.
# all_fileids[start_idx:] -> takes elements from start_idx to the end.
# all_fileids[:start_idx] -> takes elements from the beginning up to (but not including) start_idx.
# The '+' operator joins these two lists, effectively 'looping' the list around the start_idx.
reordered_fileids = all_fileids[start_idx:] + all_fileids[:start_idx]

# Initialize empty lists to store our processed data
documents = []
labels = []

# 5. Iterate through the newly ordered list
for fileid in reordered_fileids:
    # movie_reviews.raw(fileid) retrieves the full text of the review
    documents.append(movie_reviews.raw(fileid))

    # fileid is usually structured as 'label/filename.txt' (e.g., 'neg/cv000_29416.txt')
    # .split('/') divides the string into a list: ['neg', 'cv000_29416.txt']
    # [0] selects the first element, which is our category label ('pos' or 'neg')
    labels.append(fileid.split('/')[0])

# 6. Print summary statistics and a preview of the first document in this specific load
print(f"Total documents loaded: {len(documents)}")
print(f"Started loading from index: {start_idx}")
print("\nFirst review excerpt in current load:\n")
# documents[0][:300] uses string slicing to show the first 300 characters of the first review
print(documents[0][:300])

Total documents loaded: 2000
Started loading from index: 546

First review excerpt in current load:

michael richards leaves his spot as kramer on the infamous seinfeld tv sitcom for a stint as a lanky , goofy best friend to jeff daniels' lawyer character in this ill-fated , and unfunny , " comedy " . 
plot : richard the actor ( richards ) has to take the place of charles the lawyer ( daniels ) in 


TF-IDF represents a document with numbers based on which words appear and how important those words are across the full set of reviews. It gives more weight to words that help separate one review from the others instead of common words that appear everywhere.

Cell 3:
This cell converts the reviews into TF-IDF feature vectors. The output shows the size of the matrix, the sparsity level, and the top weighted terms from the first review vector.

In [3]:
# Cell 3 Build TF-IDF feature vectors
vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    ngram_range=(1, 2)
)

tfidf_matrix = vectorizer.fit_transform(documents)
feature_names = np.array(vectorizer.get_feature_names_out())

sparsity = 1.0 - (tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]))

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Sparsity: {sparsity * 100:.2f}%")

doc0_vector = tfidf_matrix[0].toarray().flatten()
top_indices = np.argsort(doc0_vector)[::-1][:10]

print("\nTop 10 TF-IDF terms for document 0:")
for term in feature_names[top_indices]:
    print(term)

TF-IDF matrix shape: (2000, 5000)
Sparsity: 96.20%

Top 10 TF-IDF terms for document 0:
charlize
richards
daniels
theron
movie
waste time
charles
lawyer
la
waste


The top terms suggest what TF-IDF considers most important in the first review The vector highlights weighted words and phrases, but it does not show the full sentence structure or full context. I would guess that this review would be labeled negative because the word waste would be considered negative.

Cell 4:
This cell defines a retrieval function and tests it on a query. (retrieval of similar items). The output shows the five most similar reviews, along with their labels and cosine similarity scores.
•	a higher score means the review uses a more similar pattern of weighted words
•	a lower score means less overlap in the important terms

We do this cell to test whether TF-IDF can retrieve reviews that look similar to the featured query.

In [4]:
# Cell 4
def find_similar_texts(query, n=5):
    query_vector = vectorizer.transform([query])
    similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()
    top_indices = np.argsort(similarities)[::-1][:n]

    results = []
    for idx in top_indices:
        text_excerpt = documents[idx][:220]
        label = labels[idx]
        similarity_score = similarities[idx]
        results.append((text_excerpt, label, similarity_score))
    return results

query = "This film was a total waste of time. Terrible acting, boring plot."
results = find_similar_texts(query, n=5)

print(f"Query: {query}\n")
for i, (text, label, score) in enumerate(results, start=1):
    print(f"Rank {i} | {label.upper()} | Score: {score:.3f}")
    print(text[:200])
    print()

Query: This film was a total waste of time. Terrible acting, boring plot.

Rank 1 | NEG | Score: 0.288
this film is extraordinarily horrendous and i'm not going to waste any more words on it . 


Rank 2 | NEG | Score: 0.228
 " nothing more than a high budget masturbation fantasy " 
showgirls ( nc-17 ) - contains graphic nudity , profanity , sexual situations and violence . 
some people , however , keep their clothes on .

Rank 3 | NEG | Score: 0.162
michael richards leaves his spot as kramer on the infamous seinfeld tv sitcom for a stint as a lanky , goofy best friend to jeff daniels' lawyer character in this ill-fated , and unfunny , " comedy " 

Rank 4 | NEG | Score: 0.157
in the line of duty is the critically praised series of television movies dealing with the real-life incidents that claimed lives of law enforcement officers in usa . 
the twilight murders , another o

Rank 5 | NEG | Score: 0.154
here i sit at my computer about to write my review of the recent action comedy " bait 

All five retrieved results are labeled negative. No positive reviews appear in the top five for this query.
It happens because the query contains strong negative terms that overlap with negative reviews in the corpus.

Cell 5:
This cell tests four queries that are meant to expose weak spots in TF-IDF retrieval. The output shows how many positive and negative reviews appear in the top five results for each query.

In [5]:
# Cell 5
failure_queries = [
    ("This movie was absolutely not a waste of time.", "Same words as the Cell 4 query, but 'not' reverses the meaning."),
    ("I watched this film three times in one weekend.", "Strongly positive, but contains zero sentiment words."),
    ("I can’t imagine anyone not enjoying this.", "Positive, but two negatives TF-IDF may misread."),
    ("A masterpiece. Stunning cinematography, a perfect score.", "Unambiguously positive: all strong positive adjectives.")
]

for query_string, note in failure_queries:
    results = find_similar_texts(query_string, n=5)
    pos_count = sum(1 for _, label, _ in results if label == 'pos')
    neg_count = sum(1 for _, label, _ in results if label == 'neg')

    print(f"Query: {query_string}")
    print(f"Note: {note}")
    print(f"Top 5 results -> POS: {pos_count}, NEG: {neg_count}")
    print("-" * 60)

Query: This movie was absolutely not a waste of time.
Note: Same words as the Cell 4 query, but 'not' reverses the meaning.
Top 5 results -> POS: 0, NEG: 5
------------------------------------------------------------
Query: I watched this film three times in one weekend.
Note: Strongly positive, but contains zero sentiment words.
Top 5 results -> POS: 3, NEG: 2
------------------------------------------------------------
Query: I can’t imagine anyone not enjoying this.
Note: Positive, but two negatives TF-IDF may misread.
Top 5 results -> POS: 1, NEG: 4
------------------------------------------------------------
Query: A masterpiece. Stunning cinematography, a perfect score.
Note: Unambiguously positive: all strong positive adjectives.
Top 5 results -> POS: 3, NEG: 2
------------------------------------------------------------


TF-IDF gets confused when the meaning depends on negation or implied sentiment instead of obvious words. It does better when the positive or negative language is direct.

Cell 6: HOG
This cell loads CIFAR-10 and selects 15 images from each class. The output displays one sample image from each class so the dataset can be checked.

In [ ]:
# Cell 6
import tensorflow as tf

(train_images, train_labels), (_, _) = tf.keras.datasets.cifar10.load_data()

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
N_PER_CLASS = 15

image_groups = []
img_labels = []
img_names = []

for class_idx, class_name in enumerate(class_names):
    class_indices = np.where(train_labels.flatten() == class_idx)[0][:N_PER_CLASS]
    class_images = train_images[class_indices]
    image_groups.append(class_images)
    img_labels.extend([class_idx] * N_PER_CLASS)
    img_names.extend([class_name] * N_PER_CLASS)

images = np.vstack(image_groups)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i * N_PER_CLASS])
    ax.set_title(class_names[i])
    ax.axis('off')

plt.tight_layout()
plt.show()

A feature vector for an image captures selected visual information in a numeric form. Instead of storing the full image meaning, it stores a reduced representation that can be compared across images.

Cell 7:
This cell converts each image into a HOG feature vector and creates a HOG visualization for each one. The output shows the HOG matrix shape, compares the number of HOG features to TF-IDF features, and displays example images in original, grayscale, and HOG form.

In [ ]:
# Cell 7 CV Start
from skimage.feature import hog
from skimage.color import rgb2gray
from skimage.transform import resize

TARGET_SIZE = 64

def image_to_hog(image):
    resized = resize(image, (TARGET_SIZE, TARGET_SIZE))
    if resized.ndim == 3 and resized.shape[2] == 3:
        grayscale_image = rgb2gray(resized)
    else:
        grayscale_image = resized

    hog_descriptor, hog_image = hog(
        grayscale_image,
        orientations=8,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        visualize=True
    )
    return hog_descriptor, hog_image

hog_descriptors = []
hog_visuals = []

for image in images:
    descriptor, hog_image = image_to_hog(image)
    hog_descriptors.append(descriptor)
    hog_visuals.append(hog_image)

hog_matrix = np.vstack(hog_descriptors)

print(f"HOG matrix shape: {hog_matrix.shape}")
print(f"TF-IDF features: {tfidf_matrix.shape[1]}")
print(f"HOG features: {hog_matrix.shape[1]}")

sample_indices = [0, N_PER_CLASS, N_PER_CLASS * 2]

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for row, idx in enumerate(sample_indices):
    resized = resize(images[idx], (TARGET_SIZE, TARGET_SIZE))
    gray = rgb2gray(resized)

    axes[row, 0].imshow(images[idx])
    axes[row, 0].set_title(f"Original: {img_names[idx]}")
    axes[row, 0].axis('off')

    axes[row, 1].imshow(gray, cmap='gray')
    axes[row, 1].set_title("Grayscale")
    axes[row, 1].axis('off')

    axes[row, 2].imshow(hog_visuals[idx], cmap='gray')
    axes[row, 2].set_title("HOG")
    axes[row, 2].axis('off')

plt.tight_layout()
plt.show()

The HOG visualization keeps edge and gradient information from the image while reducing the original image into a simpler representation. This is similar to TF-IDF in the sense that both methods reduce the original input into selected numeric features instead of keeping the full original form.


Cell 8: Finds similar images.
This cell defines image retrieval using cosine similarity on the HOG vectors. The output displays one query image and its five most similar retrieved images for each test case.

In [ ]:
# Cell 8
def find_similar_images(query_idx, n=5):
    similarities = cosine_similarity(hog_matrix[query_idx:query_idx+1], hog_matrix).flatten()
    similarities[query_idx] = -1
    top_indices = np.argsort(similarities)[::-1][:n]
    sim_scores = similarities[top_indices]
    return top_indices, sim_scores

query_indices = [0, N_PER_CLASS, N_PER_CLASS * 2]

for query_idx in query_indices:
    top_indices, sim_scores = find_similar_images(query_idx, n=5)
    query_class = img_names[query_idx]

    fig, axes = plt.subplots(1, 6, figsize=(16, 3))

    axes[0].imshow(images[query_idx])
    axes[0].set_title("QUERY")
    axes[0].axis('off')

    for i, (idx, score) in enumerate(zip(top_indices, sim_scores), start=1):
        retrieved_class = img_names[idx]
        mark = "✓" if retrieved_class == query_class else "✗"
        axes[i].imshow(images[idx])
        axes[i].set_title(f"{retrieved_class} {mark}\n{score:.2f}")
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

Only 1 retrieved image matches the same class as the query. The airplane was the only match. The results show that HOG retrieval can find visually similar images, but class mismatches still appear in the top results.

Cell 9: Failures
This cell checks the first part of the image set for top-1 retrieval failures. The output reports the total number of failures, shows the most common confused class pairs, and displays the highest-confidence failure.

In [ ]:
# Cell 9
from collections import Counter

failures = []

for query_idx in range(min(40, len(images))):
    top_indices, sim_scores = find_similar_images(query_idx, n=1)

    if img_names[top_indices[0]] != img_names[query_idx]:
        failures.append({
            'query_idx': query_idx,
            'query_class': img_names[query_idx],
            'match_idx': top_indices[0],
            'match_class': img_names[top_indices[0]],
            'score': sim_scores[0]
        })

failures = sorted(failures, key=lambda x: x['score'], reverse=True)

print(f"Total failures found: {len(failures)}")

pair_counts = Counter((f['query_class'], f['match_class']) for f in failures)
print("\nTop 5 confused class pairs:")
for (q_class, m_class), count in pair_counts.most_common(5):
    print(f"{q_class} confused with {m_class}: {count} times")

if failures:
    best_failure = failures[0]
    q_idx = best_failure['query_idx']
    m_idx = best_failure['match_idx']
    score = best_failure['score']

    q_resized = resize(images[q_idx], (TARGET_SIZE, TARGET_SIZE))
    m_resized = resize(images[m_idx], (TARGET_SIZE, TARGET_SIZE))

    q_gray = rgb2gray(q_resized)
    m_gray = rgb2gray(m_resized)

    fig, axes = plt.subplots(2, 3, figsize=(10, 7))

    axes[0, 0].imshow(images[q_idx])
    axes[0, 0].set_title(f"Query ({img_names[q_idx]})")
    axes[0, 0].axis('off')

    axes[0, 1].imshow(q_gray, cmap='gray')
    axes[0, 1].set_title("Query Grayscale")
    axes[0, 1].axis('off')

    axes[0, 2].imshow(hog_visuals[q_idx], cmap='gray')
    axes[0, 2].set_title("Query HOG")
    axes[0, 2].axis('off')

    axes[1, 0].imshow(images[m_idx])
    axes[1, 0].set_title(f"Retrieved ({img_names[m_idx]})\nScore: {score:.2f}")
    axes[1, 0].axis('off')

    axes[1, 1].imshow(m_gray, cmap='gray')
    axes[1, 1].set_title("Retrieved Grayscale")
    axes[1, 1].axis('off')

    axes[1, 2].imshow(hog_visuals[m_idx], cmap='gray')
    axes[1, 2].set_title("Retrieved HOG")
    axes[1, 2].axis('off')

    plt.tight_layout()
    plt.show()

The output shows 35 failures in the checked set, which means the top match is often a different class from the query. The confused class pairs show that HOG retrieval can strongly match images across different categories.

The most common confused pairs in this run include bird and deer, automobile and frog, and airplane and cat.

Cell 10: Comparison. This cell builds a comparison table for TF-IDF and HOG. The output compares their input type, feature dimensions, sparsity, what they encode, what they ignore, and their core weakness.

In [10]:
# Cell 10 Comparison
import pandas as pd

tfidf_sparsity = (1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100
hog_sparsity = (np.sum(hog_matrix == 0) / hog_matrix.size) * 100

comparison = pd.DataFrame({
    "Property": [
        "Input type",
        "Feature dimensions",
        "Sparsity (%)",
        "What it encodes",
        "What it ignores",
        "Core weakness"
    ],
    "TF-IDF": [
        "Text",
        tfidf_matrix.shape[1],
        f"{tfidf_sparsity:.2f}",
        "Word frequencies (which words appear, weighted by importance)",
        "Word order, context, negation, and sentence structure. For example, it can misread 'not a waste of time' because it still sees many negative words.",
        "It depends on human-designed rules about what matters, so it misses meaning outside those rules."
    ],
    "HOG": [
        "Images",
        hog_matrix.shape[1],
        f"{hog_sparsity:.2f}",
        "Edge orientations and magnitudes (brightness gradients at multiple scales)",
        "Color, larger object meaning, and scene understanding. For example, it can confuse different classes that have similar outlines or edge patterns.",
        "It also depends on human-designed rules, so it only measures part of what a human actually sees."
    ]
})

print(comparison.to_string(index=False))

          Property                                                                                                                                              TF-IDF                                                                                                                                               HOG
        Input type                                                                                                                                                Text                                                                                                                                            Images
Feature dimensions                                                                                                                                                5000                                                                                                                                              1568
      Sparsity (%)                                           

TF-IDF shows high sparsity because most values in the matrix are zero. HOG is much denser by comparison because far fewer of its values are zero.

This is directly supported by:
	•	TF-IDF sparsity: 96.20%
	•	HOG sparsity: 6.28%

TF-IDF does not keep full word order or full sentence context. HOG does not keep full color information or full object meaning. Both methods reduce the original input to selected features instead of preserving everything.

The shared limitation is that both TF-IDF and HOG are hand-designed feature systems. That means they only measure the parts of the input that the method is built to capture.

A stronger text representation would capture more sentence meaning and context. A stronger image representation would capture more object-level and scene-level meaning.

Cell 11:
This cell repeats the retrieval task from the TF-IDF section, but it replaces TF-IDF vectors with Gemini embeddings. The goal is to see whether embeddings match reviews based on broader meaning instead of mostly word overlap.

The embeddings try to capture broader meaning or semantic similarity with Geminis embedding model using the first 200 reviews. It converts each review into an embedding, stacks them into one matrix, defines a new retrieval function, then compares the TF-IDF vs embeddings on the same failure queries. The output shows that in this run the embeddings do not out perform the TF-IDF

In [11]:
# Cell 11 Bonus
import time
from google.colab import userdata
from google import genai

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

embedding_list = []

for i, text in enumerate(documents[:200]):
    truncated_text = text[:2000]
    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents=truncated_text
    )
    embedding = np.array(response.embeddings[0].values)
    embedding_list.append(embedding)

    time.sleep(0.5)
    if (i + 1) % 15 == 0:
        time.sleep(2)

embedding_matrix = np.vstack(embedding_list)

def find_similar_by_embedding(query, n=5):
    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents=query[:2000]
    )
    query_embedding = np.array(response.embeddings[0].values).reshape(1, -1)

    similarities = cosine_similarity(query_embedding, embedding_matrix).flatten()
    top_indices = np.argsort(similarities)[::-1][:n]

    results = []
    for idx in top_indices:
        text_excerpt = documents[idx][:220]
        label = labels[idx]
        similarity_score = similarities[idx]
        results.append((text_excerpt, label, similarity_score))
    return results

for query_string, _ in failure_queries:
    tfidf_results = find_similar_texts(query_string, n=3)
    embedding_results = find_similar_by_embedding(query_string, n=3)

    print(f"\nQuery: {query_string}")
    print("\nTF-IDF Results:")
    for i, (text, label, score) in enumerate(tfidf_results, start=1):
        print(f"{i}. {label.upper()} | {score:.3f} | {text[:120]}")

    print("\nEmbedding Results:")
    for i, (text, label, score) in enumerate(embedding_results, start=1):
        print(f"{i}. {label.upper()} | {score:.3f} | {text[:120]}")

    print("=" * 80)


Query: This movie was absolutely not a waste of time.

TF-IDF Results:
1. NEG | 0.348 | this film is extraordinarily horrendous and i'm not going to waste any more words on it . 

2. NEG | 0.233 | michael richards leaves his spot as kramer on the infamous seinfeld tv sitcom for a stint as a lanky , goofy best friend
3. NEG | 0.218 | starring kiefer sutherland ; reese witherspoon & bokeem woodbine i used to think that the conversation was the worse fil

Embedding Results:
1. NEG | 0.608 | well if you are up for stellar effects then this is the movie for you . 
because thats all that there really is . . . . 
2. NEG | 0.599 | this review contains spoilers , but believe me , i don't say anything you can't guess 10 minutes into the movie . 
i * d
3. NEG | 0.597 | suicide is pointless , everyone should know that . 
so what's this movie like ? 
you guessed it . . . 
pointless . 
the 

Query: I watched this film three times in one weekend.

TF-IDF Results:
1. POS | 0.138 | any movie about the

In this run, the embedding results do not clearly improve the failure queries. Several positive-sounding queries still return mostly negative reviews in the top results.

For example:
	•	“I watched this film three times in one weekend.” → embedding top 3 are all NEG
	•	“I can’t imagine anyone not enjoying this.” → embedding top 3 are all NEG
	•	“A masterpiece. Stunning cinematography, a perfect score.” → embedding top 3 are all NEG